### Workflow

This notebook reads benchmark-defined settings, studies the Tanner graph, acquires paired decoder outcomes and inspects selected BP belief histories. Run initialization and acquisition before the case analyses; trace examples explain individual inputs rather than establish an overall decoder ranking.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/research/failure_regions'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

_data_paths = ExperimentPaths(ROOT, EXPERIMENT_DIR / 'config/paths.json')


In [ ]:
from experiment.utils import misc

def aggregate_path(path):
    return misc.aggregate_path(path, data_paths=_data_paths)


In [ ]:
import numpy as np

noise_model = 'circuit_level'
d_rounds = 1
max_shots = 1000000000
max_errors = 2000
p_values = np.geomspace(1e-08, 0.1, 12)


In [ ]:
import multiprocessing
import sys
import os
from pathlib import Path
from typing import Literal
import numpy as np
import pandas as pd
import sinter
import stim
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product, combinations

DATA_DIR = _data_paths.output('main_cases')
DATA_DIR.mkdir(parents=True, exist_ok=True)

from utils.misc import *

%load_ext autoreload
%autoreload 2


In [ ]:
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD
from src.decoders import Tesseract


In [ ]:
ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}

from experiment.utils import decoder_setup

def build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=BPOSD, tesseract_type=Tesseract)


In [ ]:
from experiment.utils import benchmark_plotting

stats_to_df = benchmark_plotting.stats_to_df
plot_gap_interval = benchmark_plotting.plot_gap_interval


In [ ]:
save_file = _data_paths.input('2026_05_28_circuit_level_1_perfect_meas_shots_1000000000_csv')
stats = sinter.stats_from_csv_files(save_file)
df = stats_to_df(stats)
summary = df.pivot_table(
    index=['code', 'p'],
    columns='decoder',
    values='logical_error_rate',
    aggfunc='mean',
).reset_index().rename_axis(None, axis=1)
summary['gap_bposd_minus_tess'] = summary['BPOSD'] - summary['TESSERACT']
summary = summary.sort_values(['code', 'p']).reset_index(drop=True)


# Whole Tanner Graph Structure

In [ ]:
from utils.graph_vis import visualize_graph_plotly, flatten_graph
from utils.graph_struct import build_tanner_graph_from_dem, graph_struct


In [ ]:
error_rate = ErrorRate(p=0.01, p_meas=0.0)
builder = CircuitBuilder(ca_codes['CA_9_2_5'])
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()
TG, _ = build_tanner_graph_from_dem(dem)


In [ ]:
visualize_graph_plotly(TG)


In [ ]:
gs = graph_struct(TG)
print(gs)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
plt.subplot(1, 2, 1)
plt.scatter(range(len(gs['err_degree_dist'])), gs['err_degree_dist'])
plt.title('Error Node Degree Distribution')
plt.xticks(np.arange(0, len(gs['err_degree_dist'])))
plt.yticks(np.arange(0, max(gs['err_degree_dist']) + 1))
plt.xlabel('Error Nodes')
plt.ylabel('Degree')

plt.subplot(1, 2, 2)
plt.scatter(range(len(gs['det_degree_dist'])), gs['det_degree_dist'])
plt.title('Detector Node Degree Distribution')
plt.xticks(np.arange(0, len(gs['det_degree_dist'])))
plt.yticks(np.arange(0, max(gs['det_degree_dist']) + 1))
plt.xlabel('Detector Nodes')
plt.ylabel('Degree')
plt.tight_layout()

plt.show()


### Inspect shared parity-check neighborhoods

Extract the check matrices and inspect how variables share checks. This provides graph context for the later local failure examples without itself identifying a trapping mechanism.


In [ ]:
from beliefmatching import detector_error_model_to_check_matrices

demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
H = demMatrices.check_matrix.toarray()
L = demMatrices.observables_matrix.toarray()
p = demMatrices.priors


In [ ]:
m, n = H.shape
shared = {}
for i, j in combinations(range(m), 2):
    shared_vars = np.sum(H[i, :] & H[j, :])
    shared[i, j] = shared_vars
    print(f'Check nodes {i} and {j} share variables: {shared_vars}')


In [ ]:
np.unique_counts(list(shared.values()))


# Sample Instances

In [ ]:
max_shots = 1000000000
d_rounds = 1
noise_model = 'circuit_level'
CASES_DIR = _data_paths.output('noise_model_d_rounds_perfect_meas_shots_max_shots', noise_model=noise_model, d_rounds=d_rounds, max_shots=max_shots)
CASES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
points = summary.copy()
points = points[points['gap_bposd_minus_tess'] > 0].sort_values(['code', 'p'])
points


## Counting

In [ ]:
from utils.sample import count_samples_union


In [ ]:
cnt_path = aggregate_path(_data_paths.output('cnt_cases_csv', noise_model=noise_model, d_rounds=d_rounds, max_shots=max_shots))


In [ ]:
f = open(cnt_path, 'w')
f.write('code,p,both_success,both_failure,bposd_fail_tess_success,bposd_success_tess_fail\n')
bs_total_cnt, bf_total_cnt, bfts_total_cnt, bstf_total_cnt = (0, 0, 0, 0)
cnt_table = []

for _, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'counting cases: code={code_name}, p={p:.6f}')
    bs_cnt, bf_cnt, bfts_cnt, bstf_cnt, dem = count_samples_union(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        error_rate=ErrorRate(p=p, p_meas=0.0),
        d_rounds=d_rounds,
        total_shots=max_shots,
    )

    print(f'counted {bs_cnt} both success cases for code={code_name}, p={p:.6f}')
    print(f'counted {bf_cnt} both failure cases for code={code_name}, p={p:.6f}')
    print(f'counted {bfts_cnt} failure success cases for code={code_name}, p={p:.6f}')
    print(f'counted {bstf_cnt} success failure cases for code={code_name}, p={p:.6f}')
    print('=================================')
    bs_total_cnt += bs_cnt
    bf_total_cnt += bf_cnt
    bfts_total_cnt += bfts_cnt
    bstf_total_cnt += bstf_cnt
    f.write(f'{code_name},{p},{bs_cnt},{bf_cnt},{bfts_cnt},{bstf_cnt}\n')
    f.flush()

f.close()
print(f'saved: {cnt_path}')


### Overlap

In [ ]:
from matplotlib_venn import venn2


In [ ]:
cnt_data = pd.read_csv(cnt_path)
cnt_data


In [ ]:
fig, axes = plt.subplots(2, (cnt_data.shape[0] + 1) // 2, figsize=(48, 18))
fig.suptitle('Overlap between BPOSD Failure and Tesseract Failure cases', fontsize=24)
axes = axes.flatten()

for i, row in cnt_data.iterrows():
    if row.isnull().any():
        continue
    total_bp_fail = row['both_failure'] + row['bposd_fail_tess_success']
    total_tess_fail = row['both_failure'] + row['bposd_success_tess_fail']

    if total_bp_fail == 0 or total_tess_fail == 0:
        continue
    frac_bp = row['both_failure'] / total_bp_fail
    frac_tess = row['both_failure'] / total_tess_fail
    venn2(subsets=(row['bposd_fail_tess_success'], row['bposd_success_tess_fail'], row['both_failure']), set_labels=(f'BPOSD Fails \n({frac_bp:.2%})', f'Tesseract Fails \n({frac_tess:.2%})'), ax=axes[i])
    axes[i].set_title(f"{row['code']} (p={row['p']:.2})", fontsize=16)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

df = cnt_data[cnt_data['code'] == 'CA_9_2_5'].copy()
df['total_fail_union'] = df['both_failure'] + df['bposd_fail_tess_success'] + df['bposd_success_tess_fail']
df = df[df['total_fail_union'] > 0].copy()
df = df.sort_values('p').reset_index(drop=True)
n = len(df)

fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.8), constrained_layout=True)
if n == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, df.iterrows()):
    bp_only = int(row['bposd_fail_tess_success'])
    tess_only = int(row['bposd_success_tess_fail'])
    both = int(row['both_failure'])
    bp_fail = bp_only + both
    tess_fail = tess_only + both
    total_union = bp_only + tess_only + both
    v = venn2(subsets=(bp_only, tess_only, both), set_labels=('BP+OSD fails', 'Tesseract fails'), ax=ax)

    for label in v.set_labels:
        if label is not None:
            label.set_fontsize(9)

    for subset_id, value in [('10', bp_only), ('01', tess_only), ('11', both)]:
        label = v.get_label_by_id(subset_id)
        if label is not None:
            label.set_text(f'{value:,}')
            label.set_fontsize(9)

    bp_shared = both / bp_fail if bp_fail > 0 else np.nan
    tess_shared = both / tess_fail if tess_fail > 0 else np.nan
    ax.set_title(f"$p={row['p']:.2g}$", fontsize=13)
    ax.text(
        0.5,
        -0.08,
        f'shared / BP+OSD: {bp_shared:.1%}\nshared / Tesseract: {tess_shared:.1%}',
        transform=ax.transAxes,
        ha='center',
        va='top',
        fontsize=12,
    )

fig.suptitle('Overlap between BP+OSD and Tesseract failure sets for $d=5$', fontsize=14, y=1.08)
plt.subplots_adjust(top=0.82)
plt.show()


## Sample with errors

### Acquire labeled physical-error cases

Collect each of the four outcome groups with error-support information. The resulting files enable comparisons of sampled errors, syndromes and predictions; retained examples may be selected or capped.


In [ ]:
from utils.sample import collect_samples_with_errors


In [ ]:
cases_path = _data_paths.output('bfts_cases_csv', noise_model=noise_model, d_rounds=d_rounds, max_shots=max_shots)
cases_path.unlink(missing_ok=True)
for i, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    recs, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        error_rate=ErrorRate(p=p, p_meas=0.0),
        d_rounds=d_rounds,
        total_shots=max_shots,
        kind='bposd_fail_tess_success',
    )

    print(f'collected {len(recs)} failure cases for code={code_name}, p={p:.6f}')
    if recs:
        pd.DataFrame(recs).to_csv(cases_path, mode='a', header=not cases_path.exists(), index=False)

print('saved:', cases_path)


In [ ]:
cases_path = _data_paths.output('bs_cases_csv', noise_model=noise_model, d_rounds=d_rounds, max_shots=max_shots)
cases_path.unlink(missing_ok=True)
for i, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    recs, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        error_rate=ErrorRate(p=p, p_meas=0.0),
        d_rounds=d_rounds,
        total_shots=max_shots,
        kind='both_success',
    )

    print(f'collected {len(recs)} both_success cases for code={code_name}, p={p:.6f}')
    if recs:
        pd.DataFrame(recs).to_csv(cases_path, mode='a', header=not cases_path.exists(), index=False)

print('saved:', cases_path)


In [ ]:
cases_path = _data_paths.output('bstf_cases_csv', noise_model=noise_model, d_rounds=d_rounds, max_shots=max_shots)
cases_path.unlink(missing_ok=True)
for i, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    recs, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        error_rate=ErrorRate(p=p, p_meas=0.0),
        d_rounds=d_rounds,
        total_shots=max_shots,
        kind='bposd_success_tess_fail',
    )

    print(f'collected {len(recs)} bposd_success_tess_fail cases for code={code_name}, p={p:.6f}')
    if recs:
        pd.DataFrame(recs).to_csv(cases_path, mode='a', header=not cases_path.exists(), index=False)

print('saved:', cases_path)


In [ ]:
cases_path = _data_paths.output('bf_cases_csv', noise_model=noise_model, d_rounds=d_rounds, max_shots=max_shots)
cases_path.unlink(missing_ok=True)
for i, row in points.iterrows():
    code_name = row['code']
    p = float(row['p'])
    print(f'collecting cases: code={code_name}, p={p:.6f}')
    recs, _ = collect_samples_with_errors(
        code_name=code_name,
        ca_code=ca_codes[code_name],
        noise_model=noise_model,
        error_rate=ErrorRate(p=p, p_meas=0.0),
        d_rounds=d_rounds,
        total_shots=max_shots,
        kind='both_fail',
    )

    print(f'collected {len(recs)} both_fail cases for code={code_name}, p={p:.6f}')
    if recs:
        pd.DataFrame(recs).to_csv(cases_path, mode='a', header=not cases_path.exists(), index=False)

print('saved:', cases_path)


In [ ]:
types = {
    'detector_events': str,
    'error_events': str,
    'true_observables': str,
    'bposd_pred_observables': str,
    'tesseract_pred_observables': str,
}

bf_cases = pd.read_csv(_data_paths.output('bf_cases_csv', noise_model=noise_model, d_rounds=d_rounds, max_shots=max_shots), dtype=types)
bs_cases = pd.read_csv(_data_paths.output('bs_cases_csv', noise_model=noise_model, d_rounds=d_rounds, max_shots=max_shots), dtype=types)
bfts_cases = pd.read_csv(_data_paths.output('bfts_cases_csv', noise_model=noise_model, d_rounds=d_rounds, max_shots=max_shots), dtype=types)
bstf_cases = pd.read_csv(_data_paths.output('bstf_cases_csv', noise_model=noise_model, d_rounds=d_rounds, max_shots=max_shots), dtype=types)


### Error Weight Distribution

CA_5_2_3

In [ ]:
error_weight_df = pd.DataFrame({
    'bf_cases': bf_cases[bf_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
    'bs_cases': bs_cases[bs_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
    'bfts_cases': bfts_cases[bfts_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
    'bstf_cases': bstf_cases[bstf_cases['code'] == 'CA_5_2_3']['error_weight'].value_counts(),
}).fillna(0).astype(int)
comparison_df = error_weight_df.rename_axis(columns='case').stack().rename('count').reset_index()
comparison_df['percentage'] = comparison_df['count'] / comparison_df.groupby('case')['count'].transform('sum') * 100
comparison_df = comparison_df.pivot(index='error_weight', columns='case', values=['count', 'percentage']).swaplevel(axis=1).sort_index(axis=1, level=0)
comparison_df


In [ ]:
count_df = comparison_df.xs('count', axis=1, level=1)
percentage_df = comparison_df.xs('percentage', axis=1, level=1)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.suptitle('Error Weight Distribution for CA_5_2_3', fontsize=12)

count_df.plot(
    kind='bar',
    title='Count vs Error Weight',
    ylabel='Count',
    ax=axes[0],
)
percentage_df.plot(
    kind='bar',
    title='Percentage vs Error Weight',
    ylabel='Percentage (%)',
    ax=axes[1],
)


CA_9_2_5

In [ ]:
error_weight_df = pd.DataFrame({
    'bf_cases': bf_cases[bf_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
    'bs_cases': bs_cases[bs_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
    'bfts_cases': bfts_cases[bfts_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
    'bstf_cases': bstf_cases[bstf_cases['code'] == 'CA_9_2_5']['error_weight'].value_counts(),
}).fillna(0).astype(int)
comparison_df = error_weight_df.rename_axis(columns='case').stack().rename('count').reset_index()
comparison_df['percentage'] = comparison_df['count'] / comparison_df.groupby('case')['count'].transform('sum') * 100
comparison_df = comparison_df.pivot(index='error_weight', columns='case', values=['count', 'percentage']).swaplevel(axis=1).sort_index(axis=1, level=0)
comparison_df


In [ ]:
count_df = comparison_df.xs('count', axis=1, level=1)
percentage_df = comparison_df.xs('percentage', axis=1, level=1)
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.suptitle('Error Weight Distribution for CA_9_2_5', fontsize=12)

count_df.plot(
    kind='bar',
    title='Count vs Error Weight',
    ylabel='Count',
    ax=axes[0],
)
percentage_df.plot(
    kind='bar',
    title='Percentage vs Error Weight',
    ylabel='Percentage (%)',
    ax=axes[1],
)


In [ ]:
count_df = comparison_df.xs('count', axis=1, level=1)
gap_counts = count_df['bfts_cases']
gap_percentage = 100 * gap_counts / gap_counts.sum()

fig, ax = plt.subplots(figsize=(7.2, 4.6))
colors = ['tab:orange' if w == 2 else 'tab:blue' for w in gap_percentage.index]
ax.bar(gap_percentage.index, gap_percentage.values, width=0.7, color=colors)
ax.set_xlabel('Error weight', fontsize=12)
ax.set_ylabel('Fraction (%)', fontsize=12)
ax.legend([f'BP+OSD wrong, Tesseract correct'], fontsize=10)
ax.set_xticks(gap_percentage.index)
ax.grid(
    True,
    axis='y',
    which='major',
    alpha=0.22,
    linewidth=0.7,
)
ax.set_axisbelow(True)
plt.tight_layout()

plt.show()


### Resolve the distribution by p

Break down the selected disagreement population by p and error weight. Pooling across different p values can hide changes in which physical supports are represented.


In [ ]:
dist = bfts_cases[bfts_cases['code'] == 'CA_9_2_5'].groupby(['p', 'error_weight']).size().reset_index(name='count')
dist['prob'] = dist.groupby('p')['count'].transform(lambda x: x / x.sum())
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.suptitle('Error Weight Distribution for CA_9_2_5 in different p (bfts)', fontsize=12)

for i, (p_value, sub) in enumerate(dist.groupby('p')):
    axes[0].bar(sub['error_weight'] + i * 0.1, sub['count'], label=f'p={p_value:.6f}', width=0.1)
    axes[1].bar(sub['error_weight'] + i * 0.1, sub['prob'], label=f'p={p_value:.6f}', width=0.1)

min_w = dist['error_weight'].min()
max_w = dist['error_weight'].max()
axes[0].set_xticks(range(min_w, max_w + 1, 1))
axes[0].set_xlabel('error_weight')
axes[0].set_ylabel('count')
axes[0].set_title('count vs error weight')
axes[0].legend()
axes[1].set_xticks(range(min_w, max_w + 1, 1))
axes[1].set_xlabel('error_weight')
axes[1].set_ylabel('percentage')
axes[1].set_title('percentage vs error weight')
axes[1].legend()
plt.tight_layout()

plt.show()


In [ ]:
bfts_cases[bfts_cases['code'] == 'CA_9_2_5']


In [ ]:
plt.figure(figsize=(12, 5))
ax = sns.violinplot(
    data=bfts_cases[bfts_cases['code'] == 'CA_9_2_5'],
    x='p',
    y='error_llr_weight',
    inner='box',
)
labels = [f'{float(t.get_text()):.6e}' for t in ax.get_xticklabels()]
ax.set_xticklabels(labels, rotation=45)
plt.xticks(rotation=45)
plt.xlabel('p')
plt.ylabel('error_llr_weight')
plt.title('Distribution of error_llr_weight for different p')
plt.tight_layout()

plt.show()


# Belief Tracking

Other properties of the following four cases may be worth examining,

such as oscillation amplitude, because it affects OSD reliability.

### Prepare representative native BP traces

Select low-weight disagreement cases and initialize the Cython decoder used to inspect belief histories. Compare logical outcomes with truth separately from checking syndrome consistency.


In [ ]:
from src.decoders import CythonBPOSD
from utils.belief_tracking import plot_belief_tracking, plot_history_heatmap
from utils.graph_struct import build_tanner_graph_from_dem
from utils.graph_vis import animate_hard_decisions_on_graph, visualize_det_on_tanner_graph, visualize_graph_plotly

max_iter = 60


In [ ]:
low_weight_bfts = bfts_cases[(bfts_cases['code'] == 'CA_9_2_5') & (bfts_cases['error_weight'] == 2)]
low_weight_bfts


In [ ]:
low_weight_bfts.groupby('detector_events').agg(
    true_obs_cnt=('true_observables', 'unique'),
    bposd_pred_obs_cnt=('bposd_pred_observables', 'unique'),
    tess_pred_obs_cnt=('tesseract_pred_observables', 'unique'),
    cnt=('detector_events', 'size'),
).sort_values('cnt', ascending=False).head(20)


In [ ]:
low_weight_bfts.groupby(['detector_events', 'error_events']).size()


In [ ]:
low_weight_bfts['p'].value_counts().sort_values(ascending=True)


In [ ]:
code = ca_codes['CA_9_2_5']
error_rate = ErrorRate(p=float(low_weight_bfts['p'].value_counts().idxmax()), p_meas=0.0)


### First Case

detector_events = 11010000000010

error_events = 001000000100000000000000000000000

In [ ]:
row = low_weight_bfts[(low_weight_bfts['detector_events'] == '11010000000010') & (low_weight_bfts['error_events'] == '001000000100000000000000000000000')].iloc[0]
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()
compiled_b = CythonBPOSD(code, error_rate.p, max_iter).compile_decoder_for_dem(dem=dem)
compiled_t = Tesseract(code, error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=dem)


In [ ]:
pred_b = compiled_b.decoder.decode(str2bits(row['detector_events']))
belief_history = compiled_b.decoder.belief_history.copy()
pred_t = compiled_t.decoder.decode(str2bits(row['detector_events']))
info_b = {
    'pred_b': pred_b,
    'belief_history_shape': belief_history.shape,
    'is_bp_converged': compiled_b.decoder.is_bp_converged,
    'is_osd_satisfied': compiled_b.decoder.is_osd_satisfied,
}
info_t = {'pred_t': pred_t.astype(int)}

print('BPOSD info:', info_b)
print('Tesseract info:', info_t)


In [ ]:
hard_history = (belief_history < 0).astype(np.uint8)
fig, ax = plt.subplots(3, 1, figsize=(12, 12))
plot_belief_tracking(belief_history, ax[0])
plot_history_heatmap(belief_history, kind='belief', fig=fig, ax=ax[1])
plot_history_heatmap(hard_history, kind='hard decision', fig=fig, ax=ax[2])


In [ ]:
TG, _ = build_tanner_graph_from_dem(dem)
hard_history = (belief_history < 0).astype(np.uint8)
hard_decisions_list = list(map(lambda x: ''.join(map(str, x)), hard_history))
ani = animate_hard_decisions_on_graph(TG, row['detector_events'], hard_decisions_list, err_events=row['error_events'])

ani.show()


Fourier Transform

### Frequency content of the first trace

Analyze the time dependence of individual belief variables using the trace computed above. Spectral peaks help describe oscillation in this example, but do not establish its cause or prevalence.


In [ ]:
T, N = belief_history.shape
dt = 1.0
belief_centered = belief_history - np.mean(belief_history, axis=0, keepdims=True)
belief_fft = np.fft.rfft(belief_centered, axis=0)
frequencies = np.fft.rfftfreq(T, d=dt)
amplitude = np.abs(belief_fft) / T

if T % 2 == 0:
    amplitude[1:-1, :] *= 2
else:
    amplitude[1:, :] *= 2

plt.subplots(figsize=(12, 6))
for j in range(N):
    plt.plot(frequencies, amplitude[:, j], marker='o', label=f'var {j}')
plt.xticks(frequencies, [f'{f:.2f}' for f in frequencies])
plt.xlabel('Frequency')
plt.ylabel('Amplitude')
plt.title(f'Fourier spectrum of variables beliefs')
plt.legend()

plt.show()


In [ ]:
results = []
positive_freq_amp = amplitude[1:, :]
positive_freqs = frequencies[1:]

for j in range(N):
    signal = positive_freq_amp[:, j]
    median_signal = np.median(signal)
    max_signal = np.max(signal)
    peak_idx = np.argmax(signal)
    peak_frequency = positive_freqs[peak_idx]

    if median_signal > 1e-12:
        max_median_ratio = max_signal / median_signal
    else:
        max_median_ratio = np.inf
    results.append({
        'variable': j,
        'median': median_signal,
        'max': max_signal,
        'max/median': max_median_ratio,
        'peak_frequency': peak_frequency,
    })

df = pd.DataFrame(results)
df


In [ ]:
data = {
    'variable': list(range(33)),
    'median': [1.726311, 1.2317, 1.252849, 0.892369, 0.531589, 0.896726, 0.608422, 1.099647, 0.317801, 1.465861, 0.990674, 1.118593, 0.819985, 0.690976, 0.835163, 0.817575, 0.903156, 1.200207, 0.566198, 0.749121, 0.735657, 0.776033, 0.933506, 0.011853, 1.291641, 1.542051, 0.950835, 0.561344, 0.644256, 0.587436, 0.401204, 0.736059, 0.64093],
    'max': [1.789912, 2.584973, 2.620429, 1.166308, 0.769284, 1.848361, 1.809999, 2.555995, 0.928014, 2.597342, 2.149, 1.90845, 1.704057, 1.338215, 2.668521, 1.253608, 0.993939, 2.682537, 1.615719, 2.281477, 1.186377, 0.945014, 0.987018, 0.020097, 2.116142, 4.507277, 1.709646, 2.477227, 1.08808, 1.864391, 1.784896, 1.537756, 1.284328],
    'max/median': [1.036842, 2.098704, 2.091576, 1.30698, 1.44714, 2.061233, 2.974906, 2.324378, 2.920113, 1.771888, 2.16923, 1.706116, 2.078157, 1.936703, 3.19521, 1.533324, 1.100517, 2.235062, 2.853628, 3.045538, 1.612677, 1.21775, 1.057324, 1.695518, 1.638336, 2.922911, 1.798046, 4.41303, 1.688894, 3.173779, 4.448853, 2.089175, 2.003852],
    'peak_frequency': [0.375, 0.375, 0.375, 0.375, 0.25, 0.375, 0.375, 0.375, 0.25, 0.375, 0.375, 0.375, 0.375, 0.375, 0.375, 0.375, 0.375, 0.375, 0.375, 0.375, 0.375, 0.375, 0.25, 0.375, 0.375, 0.375, 0.375, 0.375, 0.125, 0.375, 0.375, 0.375, 0.125],
}

df = pd.DataFrame(data)


In [ ]:
metrics = ['median', 'max', 'max/median', 'peak_frequency']
ax = df.plot(
    x='variable',
    y=metrics,
    kind='line',
    title='Median and Max Amplitude of Variable Beliefs',
    ylabel='Amplitude',
    xlabel='Variable Index',
    figsize=(12, 6),
    legend=True,
    grid=True,
)
highlight_vars = [0, 1, 2, 7, 9, 13, 25, 31]

for v in highlight_vars:
    ax.axvline(
        x=v,
        color='red',
        linestyle='--',
        alpha=0.4,
        linewidth=1,
    )
highlight_df = df[df['variable'].isin(highlight_vars)]

for metric in metrics:
    ax.scatter(
        highlight_df['variable'],
        highlight_df[metric],
        s=55,
        facecolors='none',
        edgecolors='red',
        linewidths=1.5,
        zorder=10,
    )
plt.tight_layout()

plt.show()


## Second Case

detector_events = 11010000000010

error_events = 010000010000000000000000000000000

### Compare another error with the same detector pattern

Trace a different recorded error explanation for the selected syndrome. Compare belief dynamics and true logical observables; shared detectors alone do not identify the physical error.


In [ ]:
row = low_weight_bfts[(low_weight_bfts['detector_events'] == '11010000000010') & (low_weight_bfts['error_events'] == '010000010000000000000000000000000')].iloc[0]
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()
compiled_b = CythonBPOSD(code, error_rate.p, max_iter).compile_decoder_for_dem(dem=dem)
compiled_t = Tesseract(code, error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=dem)


In [ ]:
pred_b = compiled_b.decoder.decode(str2bits(row['detector_events']))
belief_history = compiled_b.decoder.belief_history.copy()
pred_t = compiled_t.decoder.decode(str2bits(row['detector_events']))
info_b = {
    'pred_b': pred_b,
    'belief_history_shape': belief_history.shape,
    'is_bp_converged': compiled_b.decoder.is_bp_converged,
    'is_osd_satisfied': compiled_b.decoder.is_osd_satisfied,
}
info_t = {'pred_t': pred_t.astype(int)}

print('BPOSD info:', info_b)
print('Tesseract info:', info_t)


In [ ]:
TG, _ = build_tanner_graph_from_dem(dem)
hard_history = (belief_history < 0).astype(np.uint8)
hard_decisions_list = list(map(lambda x: ''.join(map(str, x)), hard_history))
ani = animate_hard_decisions_on_graph(TG, row['detector_events'], hard_decisions_list, err_events=row['error_events'])

ani.show()


Fourier Transform


In [ ]:
T, N = belief_history.shape
dt = 1.0
belief_centered = belief_history - np.mean(belief_history, axis=0, keepdims=True)
belief_fft = np.fft.rfft(belief_centered, axis=0)
frequencies = np.fft.rfftfreq(T, d=dt)
amplitude = np.abs(belief_fft) / T

if T % 2 == 0:
    amplitude[1:-1, :] *= 2
else:
    amplitude[1:, :] *= 2

plt.subplots(figsize=(12, 6))
for j in range(N):
    plt.plot(frequencies, amplitude[:, j], marker='o', label=f'var {j}')
plt.xticks(frequencies, [f'{f:.2f}' for f in frequencies])
plt.xlabel('Frequency')
plt.ylabel('Amplitude')
plt.title(f'Fourier spectrum of variables beliefs')
plt.legend()

plt.show()


## Third Case

detector_events = 11011000000110

error_events = 001000010000000000000000000000000

### Repeat on the second detector pattern

Use the next detector/error pair to inspect whether the earlier trace features recur. The following Fourier analysis uses this newly computed belief history.


In [ ]:
row = low_weight_bfts[(low_weight_bfts['detector_events'] == '11011000000110') & (low_weight_bfts['error_events'] == '001000010000000000000000000000000')].iloc[0]
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()
compiled_b = CythonBPOSD(code, error_rate.p, max_iter).compile_decoder_for_dem(dem=dem)
compiled_t = Tesseract(code, error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=dem)


In [ ]:
pred_b = compiled_b.decoder.decode(str2bits(row['detector_events']))
belief_history = compiled_b.decoder.belief_history.copy()
pred_t = compiled_t.decoder.decode(str2bits(row['detector_events']))
info_b = {
    'pred_b': pred_b,
    'belief_history_shape': belief_history.shape,
    'is_bp_converged': compiled_b.decoder.is_bp_converged,
    'is_osd_satisfied': compiled_b.decoder.is_osd_satisfied,
}
info_t = {'pred_t': pred_t.astype(int)}

print('BPOSD info:', info_b)
print('Tesseract info:', info_t)


In [ ]:
TG, _ = build_tanner_graph_from_dem(dem)
hard_history = (belief_history < 0).astype(np.uint8)
hard_decisions_list = list(map(lambda x: ''.join(map(str, x)), hard_history))
ani = animate_hard_decisions_on_graph(TG, row['detector_events'], hard_decisions_list, err_events=row['error_events'])

ani.show()


Fourier Transform


In [ ]:
T, N = belief_history.shape
dt = 1.0
belief_centered = belief_history - np.mean(belief_history, axis=0, keepdims=True)
belief_fft = np.fft.rfft(belief_centered, axis=0)
frequencies = np.fft.rfftfreq(T, d=dt)
amplitude = np.abs(belief_fft) / T

if T % 2 == 0:
    amplitude[1:-1, :] *= 2
else:
    amplitude[1:, :] *= 2

plt.subplots(figsize=(12, 6))
for j in range(N):
    plt.plot(frequencies, amplitude[:, j], marker='o', label=f'var {j}')
plt.xticks(frequencies, [f'{f:.2f}' for f in frequencies])
plt.xlabel('Frequency')
plt.ylabel('Amplitude')
plt.title(f'Fourier spectrum of variables beliefs')
plt.legend()

plt.show()


In [ ]:
results = []
positive_freq_amp = amplitude[1:, :]
positive_freqs = frequencies[1:]

for j in range(N):
    signal = positive_freq_amp[:, j]
    median_signal = np.median(signal)
    max_signal = np.max(signal)
    peak_idx = np.argmax(signal)
    peak_frequency = positive_freqs[peak_idx]

    if median_signal > 1e-12:
        max_median_ratio = max_signal / median_signal
    else:
        max_median_ratio = np.inf
    results.append({
        'variable': j,
        'median': median_signal,
        'max': max_signal,
        'max/median': max_median_ratio,
        'peak_frequency': peak_frequency,
    })

df = pd.DataFrame(results)
df


In [ ]:
data = {
    'variable': list(range(33)),
    'median': [0.427709, 0.752996, 0.644526, 0.471953, 0.256365, 0.424536, 0.21376, 0.68743, 0.244424, 0.832099, 0.328946, 0.398507, 0.238871, 0.257074, 0.38836, 0.268218, 0.260401, 0.57389, 0.192079, 0.169811, 0.218949, 0.256269, 0.258042, 0.052628, 0.553838, 0.485233, 0.366786, 0.318219, 0.337274, 0.294461, 0.348714, 0.174789, 0.269779],
    'max': [1.404562, 5.600239, 6.138113, 1.506794, 1.613935, 2.333263, 0.948918, 5.387345, 0.804091, 5.101779, 2.780266, 2.838795, 0.80837, 2.83615, 2.935268, 2.36611, 1.371935, 2.572248, 1.717877, 1.149875, 0.779735, 0.492444, 1.968312, 0.092076, 3.247229, 1.969944, 2.219134, 1.83459, 3.02281, 1.888713, 1.206472, 1.43747, 2.228543],
    'max/median': [3.283917, 7.437273, 9.523449, 3.192677, 6.295457, 5.49603, 4.439171, 7.836936, 3.28974, 6.131215, 8.452053, 7.123575, 3.384137, 11.032416, 7.558109, 8.821587, 5.26855, 4.482123, 8.943618, 6.771483, 3.561266, 1.921588, 7.627864, 1.749561, 5.863136, 4.05979, 6.050209, 5.765183, 8.962486, 6.414143, 3.459773, 8.224035, 8.260633],
    'peak_frequency': [0.482759, 0.482759, 0.482759, 0.482759, 0.482759, 0.482759, 0.241379, 0.482759, 0.482759, 0.482759, 0.482759, 0.482759, 0.241379, 0.482759, 0.482759, 0.482759, 0.482759, 0.482759, 0.482759, 0.482759, 0.241379, 0.448276, 0.482759, 0.275862, 0.482759, 0.482759, 0.482759, 0.482759, 0.482759, 0.482759, 0.482759, 0.482759, 0.482759],
}

df = pd.DataFrame(data)


In [ ]:
metrics = ['median', 'max', 'max/median', 'peak_frequency']
ax = df.plot(
    x='variable',
    y=metrics,
    kind='line',
    title='Median and Max Amplitude of Variable Beliefs',
    ylabel='Amplitude',
    xlabel='Variable Index',
    figsize=(12, 6),
    legend=True,
    grid=True,
)
highlight_vars = [0, 1, 2, 7, 9, 13, 14, 17]

for v in highlight_vars:
    ax.axvline(
        x=v,
        color='red',
        linestyle='--',
        alpha=0.4,
        linewidth=1,
    )
highlight_df = df[df['variable'].isin(highlight_vars)]

for metric in metrics:
    ax.scatter(
        highlight_df['variable'],
        highlight_df[metric],
        s=55,
        facecolors='none',
        edgecolors='red',
        linewidths=1.5,
        zorder=10,
    )
plt.tight_layout()

plt.show()


## Forth Case

detector_events = 11011000000110

error_events = 010000000100000000000000000000000

### Compare the alternative explanation

Trace the other selected error support for this detector pattern and repeat the frequency analysis. The comparison stays at the level of illustrative cases.


In [ ]:
row = low_weight_bfts[(low_weight_bfts['detector_events'] == '11011000000110') & (low_weight_bfts['error_events'] == '010000000100000000000000000000000')].iloc[0]
builder = CircuitBuilder(code)
circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
dem = circ.detector_error_model()
compiled_b = CythonBPOSD(code, error_rate.p, max_iter).compile_decoder_for_dem(dem=dem)
compiled_t = Tesseract(code, error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=dem)


In [ ]:
pred_b = compiled_b.decoder.decode(str2bits(row['detector_events']))
belief_history = compiled_b.decoder.belief_history.copy()
pred_t = compiled_t.decoder.decode(str2bits(row['detector_events']))
info_b = {
    'pred_b': pred_b,
    'belief_history_shape': belief_history.shape,
    'is_bp_converged': compiled_b.decoder.is_bp_converged,
    'is_osd_satisfied': compiled_b.decoder.is_osd_satisfied,
}
info_t = {'pred_t': pred_t.astype(int)}

print('BPOSD info:', info_b)
print('Tesseract info:', info_t)


In [ ]:
TG, _ = build_tanner_graph_from_dem(dem)
hard_history = (belief_history < 0).astype(np.uint8)
hard_decisions_list = list(map(lambda x: ''.join(map(str, x)), hard_history))
ani = animate_hard_decisions_on_graph(TG, row['detector_events'], hard_decisions_list, err_events=row['error_events'])

ani.show()


Fourier Transform

In [ ]:
T, N = belief_history.shape
dt = 1.0
belief_centered = belief_history - np.mean(belief_history, axis=0, keepdims=True)
belief_fft = np.fft.rfft(belief_centered, axis=0)
frequencies = np.fft.rfftfreq(T, d=dt)
amplitude = np.abs(belief_fft) / T

if T % 2 == 0:
    amplitude[1:-1, :] *= 2
else:
    amplitude[1:, :] *= 2

plt.subplots(figsize=(12, 6))
for j in range(N):
    plt.plot(frequencies, amplitude[:, j], marker='o', label=f'var {j}')
plt.xticks(frequencies, [f'{f:.2f}' for f in frequencies])
plt.xlabel('Frequency')
plt.ylabel('Amplitude')
plt.title(f'Fourier spectrum of variables beliefs')
plt.legend()

plt.show()


### Optional historical example

Look for the final fixed historical sample in the current acquisition. If it is absent, the existing code displays an explanation and skips that example rather than changing the sample selection silently.


In [ ]:
from IPython.display import Markdown, display

historical_example = low_weight_bfts[(low_weight_bfts['detector_events'] == '00001000000110') & (low_weight_bfts['error_events'] == '010000000100000000000000000000000')]

if historical_example.empty:
    display(Markdown('The historical example is absent from the regenerated cohort; this visualization is skipped.'))
else:
    row = historical_example.iloc[0]
    builder = CircuitBuilder(code)
    circ = builder.get_circuit_level_circuit(error_rate, d_rounds=1)
    dem = circ.detector_error_model()
    compiled_b = CythonBPOSD(code, error_rate.p, max_iter).compile_decoder_for_dem(dem=dem)
    compiled_t = Tesseract(code, error_rate.p, beam=8, pq_limit=200000, det_penalty=0.0).compile_decoder_for_dem(dem=dem)

    pred_b = compiled_b.decoder.decode(str2bits(row['detector_events']))
    belief_history = compiled_b.decoder.belief_history.copy()
    pred_t = compiled_t.decoder.decode(str2bits(row['detector_events']))
    info_b = {
        'pred_b': pred_b,
        'belief_history_shape': belief_history.shape,
        'is_bp_converged': compiled_b.decoder.is_bp_converged,
        'is_osd_satisfied': compiled_b.decoder.is_osd_satisfied,
    }
    info_t = {'pred_t': pred_t.astype(int)}

    print('BPOSD info:', info_b)
    print('Tesseract info:', info_t)

    TG, _ = build_tanner_graph_from_dem(dem)
    hard_history = (belief_history < 0).astype(np.uint8)
    hard_decisions_list = list(map(lambda x: ''.join(map(str, x)), hard_history))
    ani = animate_hard_decisions_on_graph(TG, row['detector_events'], hard_decisions_list, err_events=row['error_events'])

    ani.show()
